# Produce MSI Mouse Brain Prediction CSVs

In [2]:
import sys
from pathlib import Path

start = Path.cwd().resolve()
for candidate in (start, *start.parents):
    sprint_dir = candidate / "codes" / "sprint"
    if (sprint_dir / "prediction_export.py").exists():
        PROJECT_ROOT = candidate
        if str(PROJECT_ROOT) not in sys.path:
            sys.path.insert(0, str(PROJECT_ROOT))
        break
else:
    raise RuntimeError(f"Cannot find codes/sprint/prediction_export.py from {start}")

LEGACY_DIR = PROJECT_ROOT / "codes" / "_legacy_models" / "msi_mouse_brain"
if str(LEGACY_DIR) not in sys.path:
    sys.path.insert(0, str(LEGACY_DIR))

from codes.sprint.prediction_export import select_least_used_cuda_before_torch_import

select_least_used_cuda_before_torch_import()

Detected CUDA devices before torch import:
  physical=0 used=1046 MB / 11264 MB (9.3%) <-- selected as cuda:0


In [3]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import torch
from torch.utils.data import DataLoader

from codes.sprint.prediction_export import export_msi_model_specs, find_project_root

# Legacy model & dataset classes
from codes._legacy_models.msi_mouse_brain.model import SelfAttentionMSIModel
from codes._legacy_models.msi_mouse_brain.utils import MouseBrainMSIDataset, set_seed

PROJECT_ROOT = find_project_root(Path.cwd())
print(f"Project root: {PROJECT_ROOT}")

set_seed(42)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Project root: /media/yuzhi/My PSSD/spatialProtein/spatialProtein/Github
Using device: cuda:0


In [4]:
# ---- Paths ----
TRAIN_H5AD = PROJECT_ROOT / "datas" / "msi" / "mouse_brain_1" / "mouse_brain_1_Processed.h5ad"
VAL_H5AD = PROJECT_ROOT / "datas" / "msi" / "mouse_brain_2" / "mouse_brain_2_Processed.h5ad"
MODEL_SAVE_ROOT = PROJECT_ROOT /"datas"/ "models" / "msi_mouse_brain"
OUTPUT_DIR = PROJECT_ROOT /"datas"/ "outputs" / "msi_mouse_brain"

MODEL_SPECS = [
    {
        "label": "A2",
        "scheme": "A2",
        "model_dir_candidates": ["A2"],
        "output_prefix": "msi_A2",
        "freeze_image": False,
    }
]
BATCH_SIZE = 8

for required_path in [TRAIN_H5AD, VAL_H5AD, MODEL_SAVE_ROOT]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
ad_train = sc.read_h5ad(TRAIN_H5AD)
names = [str(x) for x in list(ad_train.uns["metabolite_names"])]
y_train = ad_train.obsm["metabolite_expression_log"]
y_train = y_train.values if hasattr(y_train, "values") else y_train
y_train = np.asarray(y_train, dtype=np.float32)
target_mean = y_train.mean(axis=0).astype(np.float32)
target_std = np.maximum(y_train.std(axis=0), 1e-3).astype(np.float32)
num_genes = ad_train.n_vars
num_metabolites = len(names)
del ad_train, y_train

val_ds = MouseBrainMSIDataset(
    str(VAL_H5AD),
    names,
    target_mean,
    target_std,
    image_norm=True,
)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Inference config: {num_genes} genes, {num_metabolites} metabolites, "
      f"{len(val_ds)} validation spots")
print(names)

Inference config: 32285 genes, 1538 metabolites, 3098 validation spots
['156.26059', '156.26195', '156.48344', '156.4838', '156.485', '157.54681', '158.41217', '160.02547', '170.45622', '170.45738', '170.85749', '172.43579', '174.41102', '174.41175', '174.41199', '174.74559', '174.74624', '175.79142', '175.79277', '175.79445', '176.04346', '176.04383', '178.41135', '178.41203', '178.74656', '179.08329', '179.08397', '180.40835', '180.409', '180.74289', '180.74358', '183.75563', '183.75626', '184.0874', '184.08799', '184.41564', '184.41614', '185.0806', '185.08132', '185.41508', '185.41581', '185.75256', '185.75329', '189.74745', '189.74812', '191.08451', '191.08528', '191.41903', '191.4198', '198.01421', '209.29101', '209.29263', '209.6922', '209.69272', '213.06912', '214.09129', '214.49241', '214.49296', '214.89735', '214.89784', '215.29923', '216.48752', '216.48932', '216.8887', '216.88926', '220.50383', '220.50422', '220.90188', '221.29554', '221.29605', '222.09406', '222.09595', '2

In [6]:
saved_df, pred_df, target_df = export_msi_model_specs(
    MODEL_SPECS,
    MODEL_SAVE_ROOT,
    OUTPUT_DIR,
    val_loader,
    device,
    SelfAttentionMSIModel,
    num_genes,
    num_metabolites,
    names,
    target_mean,
    target_std,
)

display(saved_df)
display(pred_df.head())
display(target_df.head())

[A2] saved predictions: /media/yuzhi/My PSSD/spatialProtein/spatialProtein/Github/datas/outputs/msi_mouse_brain/msi_A2_predictions.csv
[A2] saved targets:     /media/yuzhi/My PSSD/spatialProtein/spatialProtein/Github/datas/outputs/msi_mouse_brain/msi_A2_targets.csv


,Model,Weights,ModelDir,Predictions,Targets,Rows,TargetsCount
0,A2,/media/yuzhi/My PSSD/spatialProtein/spatialPro...,A2,/media/yuzhi/My PSSD/spatialProtein/spatialPro...,/media/yuzhi/My PSSD/spatialProtein/spatialPro...,3098,1538


,test_index,156.26059,156.26195,156.48344,156.4838,156.485,157.54681,158.41217,160.02547,170.45622,...,1044.29885,1044.3523,1046.31366,1046.32671,1046.33977,1046.36442,1048.32025,1048.3319,1048.34354,1048.38431
0,0,4.408193,3.486995,2.624363,2.946073,4.097017,4.224545,4.879525,4.587847,6.521310,...,3.023190,2.010599,2.332760,2.641178,3.813495,2.112448,1.677795,3.308274,2.961483,1.711500
1,1,3.756998,3.293028,3.751207,3.478822,4.108495,4.268939,4.623346,4.940727,6.511699,...,3.559186,1.426123,2.708208,2.715394,2.857163,2.152621,1.596856,3.514007,3.660790,1.656386
2,2,4.813845,2.890543,2.926909,3.450781,4.756509,4.062775,4.351165,5.384767,7.059433,...,2.899607,1.285666,2.091924,2.830792,3.748684,1.897761,1.563684,3.235994,3.349208,2.165224
3,3,4.664809,3.033156,3.458006,3.633406,3.784190,3.991686,4.373535,4.374079,7.080942,...,3.502739,1.666916,2.391852,2.795171,3.124972,3.138646,1.613382,3.459180,3.544366,1.910520
4,4,4.824821,3.208080,3.450300,2.777858,4.305411,3.855804,3.844072,4.114891,7.276487,...,3.082437,2.062727,2.621106,2.659518,3.242175,2.475348,2.621610,3.179059,3.353399,1.252982


,test_index,156.26059,156.26195,156.48344,156.4838,156.485,157.54681,158.41217,160.02547,170.45622,...,1044.29885,1044.3523,1046.31366,1046.32671,1046.33977,1046.36442,1048.32025,1048.3319,1048.34354,1048.38431
0,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.219420,9.351426,8.855377,...,9.438021,0.000000,0.000000,0.000000,0.000000,10.415591,8.393681,0.000000,0.000000,0.0
1,1,5.671318,7.830028,0.000000,5.649063,0.000000,10.481077,0.000000,6.585184,7.937821,...,0.000000,6.818668,0.000000,0.000000,0.000000,0.000000,8.746112,8.744266,7.232050,0.0
2,2,8.651432,0.000000,8.542587,0.000000,9.399864,0.000000,0.000000,9.551298,7.215525,...,0.000000,0.000000,0.000000,9.852933,10.102499,0.000000,0.000000,0.000000,0.000000,0.0
3,3,9.432633,0.000000,0.000000,0.000000,0.000000,10.570237,0.000000,0.000000,9.488943,...,11.506206,0.000000,8.489126,10.416585,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
4,4,5.793401,0.000000,0.000000,5.583894,0.000000,0.000000,9.959976,0.000000,10.676600,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,11.098362,11.464041,9.829432,0.0
